# Conditional Probability and Bayes' Theorem

Notebook 09 asked: what is `P(A AND B)` when events are independent? This notebook asks the harder question: what if they're **not** independent?

**Conditional probability** answers: given that B already happened, how likely is A now? Knowing B restricts the space of possibilities — you are no longer asking "out of all outcomes" but "out of the outcomes where B is true".

This is the same operation as a `WHERE` clause in SQL. Instead of `SELECT * FROM outcomes`, you run `SELECT * FROM outcomes WHERE B = true`. Probability within that filtered view is conditional probability.

**Bayes' theorem** then inverts conditional probability: if you can measure `P(evidence | hypothesis)`, it tells you how to compute `P(hypothesis | evidence)`. This is the foundation of spam filters, medical testing, and Bayesian inference in machine learning.

**Prerequisites:** Notebooks 08 and 09 — Probability Basics, Compound Events and Independence.

**This is part 3 of 3.**

---

## Conditional Probability: A Filtered Query

The probability of A given B is written $P(A \mid B)$. Read: "P of A given B".

The definition:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

In code: `len(A & B) / len(B)` — count the outcomes matching both conditions, divided by the count of outcomes where B holds.

The denominator changes. You are no longer dividing by the full sample space; you are dividing by `B`, because B is now your restricted world.

In [ ]:
# Build the deck
suits  = ['H', 'D', 'C', 'S']
values = ['2','3','4','5','6','7','8','9','10','J','Q','K','A']
deck   = {(v, s) for v in values for s in suits}

red_cards  = {card for card in deck if card[1] in ('H', 'D')}
face_cards = {card for card in deck if card[0] in ('J', 'Q', 'K')}

def conditional_P(A, B):
    """P(A | B): probability of A given B has occurred."""
    return len(A & B) / len(B)

# P(Face | Red): among red cards only, what fraction are face cards?
p_face_given_red = conditional_P(face_cards, red_cards)

# Verify using the formula: P(F ∩ R) / P(R)
p_face_and_red  = len(face_cards & red_cards) / len(deck)
p_red           = len(red_cards) / len(deck)
p_formula       = p_face_and_red / p_red

print(f"Red cards total:   {len(red_cards)}")
print(f"Red face cards:    {len(face_cards & red_cards)}  (J, Q, K of Hearts and Diamonds)")
print()
print(f"P(Face | Red) directly:    {len(face_cards & red_cards)}/{len(red_cards)} = {p_face_given_red:.4f}")
print(f"P(Face | Red) via formula: {p_face_and_red:.4f} / {p_red:.4f} = {p_formula:.4f}")
print()
print(f"P(Face) unconditional:     {len(face_cards)}/{len(deck)} = {len(face_cards)/len(deck):.4f}")
print()
print("P(Face | Red) = P(Face) — conditioning on Red changed nothing.")
print("This confirms what notebook 09 showed: red and face are independent.")

When events are independent, conditioning on B doesn't change the probability of A. That's actually the definition of independence rewritten:

$$\text{A and B are independent} \iff P(A \mid B) = P(A)$$

Now let's look at a case where conditioning *does* matter.

In [ ]:
# P(Ace | Face card): given the card is a face card, is it an ace?
# Aces are NOT face cards — so this should be 0.
aces = {card for card in deck if card[0] == 'A'}

print(f"P(Ace | Face) = {conditional_P(aces, face_cards):.4f}")
print(f"  (Aces and face cards are mutually exclusive — they can't both be true)")
print()

# P(Heart | Red): given the card is red, is it a heart?
hearts = {card for card in deck if card[1] == 'H'}

p_heart_given_red = conditional_P(hearts, red_cards)
p_heart           = len(hearts) / len(deck)

print(f"P(Heart)        = {len(hearts)}/{len(deck)} = {p_heart:.4f}")
print(f"P(Heart | Red)  = {len(hearts & red_cards)}/{len(red_cards)} = {p_heart_given_red:.4f}")
print()
print("Knowing the card is red doubles the probability it's a heart.")
print("Red restricts to 26 cards; hearts are half of those.")

---

## Dependent Events: Drawing Without Replacement

When you draw a card and don't put it back, the second draw depends on the first. The sample space shrinks, and so do the counts.

This is the classic dependent-event scenario: the conditional probability of the second event changes based on what happened in the first.

In [ ]:
# P(both cards are aces, drawn without replacement)
p_first_ace  = 4 / 52          # 4 aces in 52 cards
p_second_ace = 3 / 51          # given first was an ace: 3 aces left in 51 remaining cards
p_both_aces  = p_first_ace * p_second_ace

print("Drawing two cards without replacement:")
print(f"  P(1st is ace)                    = 4/52  = {p_first_ace:.4f}")
print(f"  P(2nd is ace | 1st was ace)      = 3/51  = {p_second_ace:.4f}")
print(f"  P(both aces)                     = {p_first_ace:.4f} × {p_second_ace:.4f} = {p_both_aces:.6f}")
print()

# Compare: WITH replacement (independent)
p_both_aces_with_replacement = (4/52) * (4/52)
print(f"With replacement (independent):    = {p_both_aces_with_replacement:.6f}")
print(f"Without is lower — the pool shrinks after the first ace is removed.")

In [ ]:
# Enumerate directly to verify the without-replacement case
from itertools import permutations

# All ordered pairs of distinct cards (order matters for without-replacement draws)
two_card_draws = list(permutations(deck, 2))

both_aces_draws = [(c1, c2) for c1, c2 in two_card_draws
                   if c1[0] == 'A' and c2[0] == 'A']

p_exact = len(both_aces_draws) / len(two_card_draws)
print(f"Total ordered pairs:    {len(two_card_draws)}")
print(f"Both-ace pairs:         {len(both_aces_draws)}")
print(f"P(both aces) exact:     {p_exact:.6f}")
print(f"Formula result:         {p_both_aces:.6f}")
print(f"Match: {abs(p_exact - p_both_aces) < 1e-10}")

---

## The General Multiplication Rule

The multiplication rule for dependent events uses conditional probability:

$$P(A \cap B) = P(A) \cdot P(B \mid A)$$

For independent events, $P(B \mid A) = P(B)$, so this collapses to $P(A) \cdot P(B)$ — which is what notebook 09 used.

This rule chains. For three events:

$$P(A \cap B \cap C) = P(A) \cdot P(B \mid A) \cdot P(C \mid A \cap B)$$

In [ ]:
# P(first three cards drawn are all aces, without replacement)
p_first  = 4 / 52   # 4 aces from 52
p_second = 3 / 51   # 3 aces from 51
p_third  = 2 / 50   # 2 aces from 50

p_three_aces = p_first * p_second * p_third

print("P(first three draws are all aces, no replacement):")
print(f"  = (4/52) × (3/51) × (2/50)")
print(f"  = {p_first:.4f} × {p_second:.4f} × {p_third:.4f}")
print(f"  = {p_three_aces:.8f}")
print(f"  ≈ 1 in {1/p_three_aces:,.0f}")

---

## Bayes' Theorem: Updating Beliefs with Evidence

Conditional probability answers "given B, how likely is A?"

Bayes' theorem solves the inverse problem: you know `P(evidence | hypothesis)`, and you want `P(hypothesis | evidence)`.

This inversion is non-trivial. The direction matters enormously:

- `P(positive test | disease)` — the test's accuracy — is something a lab can measure.
- `P(disease | positive test)` — what you actually want to know — requires more.

Bayes' theorem connects them:

$$P(H \mid E) = \frac{P(E \mid H) \cdot P(H)}{P(E)}$$

Where:
- $P(H)$ — **prior**: how likely was H before we saw the evidence?
- $P(E \mid H)$ — **likelihood**: how likely is the evidence if H is true?
- $P(E)$ — **marginal**: overall probability of the evidence (regardless of H)
- $P(H \mid E)$ — **posterior**: how likely is H *given* the evidence?

The prior is the key ingredient that people forget. Evidence doesn't exist in a vacuum — it updates an existing belief.

### Deriving the theorem

It follows directly from the definition of conditional probability, applied twice:

$$P(H \mid E) = \frac{P(H \cap E)}{P(E)} \qquad \text{and} \qquad P(E \mid H) = \frac{P(H \cap E)}{P(H)}$$

Rearrange the second: $P(H \cap E) = P(E \mid H) \cdot P(H)$. Substitute into the first:

$$P(H \mid E) = \frac{P(E \mid H) \cdot P(H)}{P(E)}$$

That's it. It's just conditional probability applied twice and rearranged.

### The disease test: where intuition fails

A disease affects 1% of the population. A test is 99% accurate:
- 99% chance of a positive result if you have the disease (true positive rate)
- 1% chance of a positive result if you don't have the disease (false positive rate)

You test positive. What is the probability you actually have the disease?

Most people say roughly 99%. The correct answer is much lower — and understanding why is one of the most important lessons in probabilistic reasoning.

In [ ]:
# Bayes' theorem: P(Disease | Positive) = P(Positive | Disease) × P(Disease) / P(Positive)

p_disease         = 0.01    # 1% of population has it  ← the prior
p_no_disease      = 1 - p_disease

p_pos_given_dis   = 0.99    # true positive rate
p_pos_given_nodis = 0.01    # false positive rate

# Total probability of testing positive (law of total probability)
# = P(positive AND have disease) + P(positive AND no disease)
p_positive = (p_pos_given_dis   * p_disease +
              p_pos_given_nodis * p_no_disease)

# Bayes update
p_disease_given_pos = (p_pos_given_dis * p_disease) / p_positive

print("Inputs:")
print(f"  P(disease)               = {p_disease:.2%}   <- prior")
print(f"  P(positive | disease)    = {p_pos_given_dis:.2%}   <- true positive rate")
print(f"  P(positive | no disease) = {p_pos_given_nodis:.2%}   <- false positive rate")
print()
print("Calculation:")
print(f"  P(positive)              = {p_positive:.4f}")
print(f"    = ({p_pos_given_dis} × {p_disease}) + ({p_pos_given_nodis} × {p_no_disease})")
print(f"    = {p_pos_given_dis * p_disease:.4f} + {p_pos_given_nodis * p_no_disease:.4f}")
print()
print(f"  P(disease | positive)    = {p_disease_given_pos:.2%}")
print()
print("Despite a 99% accurate test, a positive result means only ~50% chance of disease.")
print("The low prior (1%) dilutes the evidence.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualise how the posterior changes as the prior changes
# Same test accuracy, different disease prevalence

p_pos_given_dis   = 0.99
p_pos_given_nodis = 0.01

priors = np.linspace(0.001, 0.5, 500)

def posterior(prior):
    p_pos = p_pos_given_dis * prior + p_pos_given_nodis * (1 - prior)
    return (p_pos_given_dis * prior) / p_pos

posteriors = [posterior(p) for p in priors]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(priors * 100, [p * 100 for p in posteriors], color='steelblue', linewidth=2)
ax.plot([1], [posterior(0.01) * 100], 'ro', markersize=10,
        label=f'1% prevalence → {posterior(0.01):.1%} posterior')
ax.plot([10], [posterior(0.10) * 100], 'gs', markersize=10,
        label=f'10% prevalence → {posterior(0.10):.1%} posterior')
ax.plot([30], [posterior(0.30) * 100], 'm^', markersize=10,
        label=f'30% prevalence → {posterior(0.30):.1%} posterior')

ax.axline((0, 0), slope=1, color='gray', linestyle='--', alpha=0.4, label='y = x (naive intuition)')
ax.set_xlabel('Prior probability of disease (%)', fontsize=11)
ax.set_ylabel('Posterior P(disease | positive test) (%)', fontsize=11)
ax.set_title('The posterior depends heavily on the prior\n(test accuracy fixed: 99% TPR, 1% FPR)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Verify with a concrete population of 100,000 people
population     = 100_000
prevalence     = 0.01

have_disease   = int(population * prevalence)         # 1,000
no_disease     = population - have_disease            # 99,000

true_positives  = int(have_disease * 0.99)            # sick + positive test
false_negatives = have_disease - true_positives       # sick + negative test
false_positives = int(no_disease  * 0.01)             # healthy + positive test
true_negatives  = no_disease - false_positives        # healthy + negative test

total_positives = true_positives + false_positives
p_disease_given_pos = true_positives / total_positives

print(f"Population: {population:,}")
print(f"  Have disease:     {have_disease:,}")
print(f"  Don't:            {no_disease:,}")
print()
print(f"Of those who test positive:")
print(f"  True positives:   {true_positives:,}  (sick, correct positive)")
print(f"  False positives:  {false_positives:,}  (healthy, wrong positive)")
print(f"  Total positives:  {total_positives:,}")
print()
print(f"P(disease | positive) = {true_positives}/{total_positives} = {p_disease_given_pos:.2%}")
print()
print("The false positive rate × the large healthy population floods the true positives.")

The population breakdown makes it viscerally clear why the result is counterintuitive: 990 true positives vs 990 false positives. Equal numbers on each side, because a 1% false positive rate applied to 99,000 healthy people produces as many false alarms as a 99% true positive rate applied to 1,000 sick people.

The prior (prevalence) is what breaks the tie.

### Sequential Bayes: updating with multiple pieces of evidence

One of Bayes' most powerful properties: the posterior from one update becomes the prior for the next. Evidence accumulates.

In [ ]:
# Same person takes the test twice. Both come back positive.
# (Assume the two tests are independent — different labs, different methods.)

def bayes_update(prior, p_pos_given_dis=0.99, p_pos_given_nodis=0.01):
    """Apply one positive test result and return the updated posterior."""
    p_pos = p_pos_given_dis * prior + p_pos_given_nodis * (1 - prior)
    return (p_pos_given_dis * prior) / p_pos

prior      = 0.01   # 1% disease prevalence

posterior1 = bayes_update(prior)
posterior2 = bayes_update(posterior1)   # first posterior becomes new prior

print(f"Prior (disease prevalence):      {prior:.2%}")
print(f"After 1st positive test:         {posterior1:.2%}")
print(f"After 2nd positive test:         {posterior2:.2%}")
print()
print("Two positive tests on the same person shifts the probability dramatically.")
print("Each test independently updates the same belief.")

---

## Exercises

1. Using the deck, compute `P(Red | Face card)`. Is this the same as `P(Face | Red)`? Why or why not? What does asymmetry in conditional probability tell you?
2. A bag has 4 red balls and 6 blue balls. You draw two without replacement. Compute `P(both red)` using the multiplication rule for dependent events. Verify by enumerating all pairs.
3. A spam filter classifies emails. 30% of incoming emails are spam. The filter correctly identifies spam 95% of the time (true positive). It incorrectly flags a legitimate email 2% of the time (false positive). An email is flagged as spam — what is the actual probability it is spam?
4. In exercise 3, what prior probability of spam would make the posterior exactly 95%? (Hint: set up the equation and solve, or search numerically.)

In [ ]:
# Your experiments here


---

## Formal Notation

**Conditional probability:**
$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}, \quad P(B) > 0$$

**General multiplication rule:**
$$P(A \cap B) = P(A) \cdot P(B \mid A)$$

**Law of total probability** (decomposing $P(E)$ over a partition $\{H, H^c\}$):
$$P(E) = P(E \mid H) \cdot P(H) + P(E \mid H^c) \cdot P(H^c)$$

**Bayes' theorem:**
$$P(H \mid E) = \frac{P(E \mid H) \cdot P(H)}{P(E)}$$

Substituting the law of total probability into the denominator:
$$P(H \mid E) = \frac{P(E \mid H) \cdot P(H)}{P(E \mid H) \cdot P(H) + P(E \mid H^c) \cdot P(H^c)}$$

**Independence (restated):**
$$A \perp B \iff P(A \mid B) = P(A) \iff P(A \cap B) = P(A) \cdot P(B)$$

---

## Real-World Connection

- **Spam filters**: Naive Bayes classifiers compute `P(spam | words in email)` by applying Bayes' theorem to word frequencies. Each word is treated as independent evidence that updates the spam probability. One of the earliest practical ML algorithms, built entirely from conditional probability.
- **A/B testing and p-values**: the p-value is `P(results this extreme | null hypothesis is true)`. Notice the direction — it is the likelihood, not the posterior. Misreading it as `P(null hypothesis | these results)` is a classic Bayesian inversion error that leads to false conclusions.
- **Medical screening**: the disease test example is not hypothetical. Low-prevalence screening programmes (asymptomatic populations) consistently produce majority-false-positive result sets. Understanding Bayes is why doctors order confirmatory tests.
- **Bayesian machine learning**: Bayesian neural networks, Gaussian processes, and probabilistic programming all formalise the prior/likelihood/posterior framework. The concepts in this notebook are the foundation.
- **Search and recommendation**: systems that surface results based on user behaviour use Bayesian updating — every click is a piece of evidence that updates `P(user interested in topic)`.

---

## Summary

- **Conditional probability** `P(A | B)` is a filtered query: probability of A within the restricted world where B is true. Formula: `P(A ∩ B) / P(B)`.
- Conditioning on B changes `P(A)` unless A and B are independent — in which case `P(A | B) = P(A)`.
- **Dependent events**: when the outcome of one draw affects the next (e.g. without replacement), use `P(A ∩ B) = P(A) × P(B | A)`.
- **Bayes' theorem** inverts conditional probability: it turns `P(evidence | hypothesis)` into `P(hypothesis | evidence)` by incorporating the prior.
- The **prior matters enormously**. A highly accurate test produces mostly false positives when the condition is rare — because the false positive rate applied to the large healthy population overwhelms the true positives.
- **Sequential updating**: the posterior from one piece of evidence becomes the prior for the next. Bayes is a framework for accumulating evidence over time.